In [1]:
!pip install pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 47.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 72.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 103.2 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pdfplumber]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
"""
HDFC Bank Statement -> Parse transactions into:
Date, Narration, Ref, ValueDate, Debit, Credit, Balance

No narration keywords.
Fixes same-date out-of-order transactions (SWEEP/INT etc.) by solving balance chain per date.
Avoids footer/summary junk from being merged into last transaction.
"""

import re
import pdfplumber

PDF_FILE = "/content/Unlocked_Acct.pdf"

# -----------------------------
# Regex Patterns
# -----------------------------
DATE_START_RE = re.compile(r"^\d{2}/\d{2}/\d{2}\b")  # dd/mm/yy at line start

# Final numeric block: ValueDate Amount ClosingBalance
AMT_BLOCK_RE = re.compile(
    r"(?P<valdate>\d{2}/\d{2}/\d{2})\s+"
    r"(?P<amount>-?\d[\d,]*\.\d{2})\s+"
    r"(?P<balance>-?\d[\d,]*\.\d{2})\b"
)

# Stop/ignore footer/summary blocks (so they don't get appended)
SUMMARY_STOP_RE = re.compile(
    r"(STATEMENT\s*SUMMARY|STATEMENTSUMMARY|GeneratedOn:|GeneratedBy:|"
    r"computer\s+generated\s+statement|does\s+not\s+require\s+signature|"
    r"RegisteredOfficeAddress:|Registered\s+Office\s+Address:|"
    r"Contents\s+of\s+this\s+statement|HDFC\s*BANK\s*LIMITED)",
    re.IGNORECASE
)

# Opening balance (from summary line) - we read it but DO NOT output that text anywhere
OPENING_BAL_RE = re.compile(
    r"OpeningBalance\s*(?P<bal>-?\d[\d,]*\.\d{2})",
    re.IGNORECASE
)

# -----------------------------
# Helpers
# -----------------------------
def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def to_float(s: str):
    if s is None:
        return None
    s = s.replace(",", "").strip()
    try:
        return float(s)
    except Exception:
        return None

def fmt_amount(x):
    if x is None or x == "":
        return ""
    if isinstance(x, (int, float)):
        return f"{x:,.2f}"
    return str(x)

def merge_wrapped_lines(text: str) -> list[str]:
    """
    Merge wrapped lines into one transaction line.
    Also skip footer/summary so it doesn't stick to last txn.
    """
    merged = []
    stop = False

    for raw in (text or "").splitlines():
        line = normalize_spaces(raw)
        if not line:
            continue

        if SUMMARY_STOP_RE.search(line):
            stop = True
        if stop:
            continue

        if DATE_START_RE.match(line):
            merged.append(line)
        else:
            if merged:
                merged[-1] = normalize_spaces(merged[-1] + " " + line)

    return merged

def pull_tail_into_narration(txn_line: str) -> str:
    """
    If extra text exists AFTER the last (ValueDate Amount Balance) block,
    move it into narration BEFORE ref+valuedate block.
    """
    line = normalize_spaces(txn_line)
    matches = list(AMT_BLOCK_RE.finditer(line))
    if not matches:
        return line

    last = matches[-1]
    tail = normalize_spaces(line[last.end():])
    if not tail:
        return line

    before = normalize_spaces(line[:last.start()])
    parts = before.split(" ")
    if len(parts) < 3:
        return line

    ref = parts[-1]
    prefix = " ".join(parts[:-1])  # date + narration

    valdate = last.group("valdate")
    amount = last.group("amount")
    balance = last.group("balance")

    return normalize_spaces(f"{prefix} {tail} {ref} {valdate} {amount} {balance}")

def parse_txn_line(line: str):
    """
    Parse merged+fixed line into fields:
    Txn_Date, Narration, Ref, Value_Date, Amount_Token, Closing_Balance
    """
    line = normalize_spaces(line)
    ms = list(AMT_BLOCK_RE.finditer(line))
    if not ms:
        return None

    last = ms[-1]
    val_date = last.group("valdate")
    amt_token = to_float(last.group("amount"))
    bal = to_float(last.group("balance"))

    before = normalize_spaces(line[:last.start()])  # TxnDate + narration + ref
    parts = before.split(" ")
    if len(parts) < 2:
        return None

    txn_date = parts[0]
    ref = parts[-1]
    narration = normalize_spaces(" ".join(parts[1:-1]))

    return {
        "Txn_Date": txn_date,
        "Narration": narration,
        "Ref": ref,
        "Value_Date": val_date,
        "Amount_Token": amt_token,
        "Closing_Balance": bal,
        "Raw": line,
    }

def delta_ok(prev_bal, curr_bal, amt_token, tol=0.50):
    if prev_bal is None or curr_bal is None or amt_token is None:
        return False
    delta = round(curr_bal - prev_bal, 2)
    return abs(abs(delta) - abs(amt_token)) <= tol

def classify(prev_bal, txn, tol=0.50):
    """
    Return (debit_str, credit_str, new_prev_bal, ok_bool)
    """
    if not delta_ok(prev_bal, txn["Closing_Balance"], txn["Amount_Token"], tol=tol):
        return ("", "", txn["Closing_Balance"], False)

    delta = round(txn["Closing_Balance"] - prev_bal, 2)
    if delta < 0:
        return (fmt_amount(abs(delta)), "", txn["Closing_Balance"], True)
    if delta > 0:
        return ("", fmt_amount(delta), txn["Closing_Balance"], True)
    return ("", "", txn["Closing_Balance"], False)

def solve_group_order(prev_bal, group, tol=0.50):
    """
    Find an order of 'group' (same date) so that all txns match balance deltas.
    Backtracking with memoization. Group size is usually small.
    Returns (ordered_list, ok_bool).
    """
    n = len(group)
    if n <= 1:
        return (group, True)

    # bitmask memo: (rounded_prev_bal, mask) -> False means dead end
    memo = set()

    def key(pb, mask):
        if pb is None:
            return ("NONE", mask)
        return (round(pb, 2), mask)

    def dfs(pb, mask, path):
        k = key(pb, mask)
        if k in memo:
            return None

        if mask == (1 << n) - 1:
            return path

        for i in range(n):
            if mask & (1 << i):
                continue
            txn = group[i]
            if delta_ok(pb, txn["Closing_Balance"], txn["Amount_Token"], tol=tol):
                new_pb = txn["Closing_Balance"]
                res = dfs(new_pb, mask | (1 << i), path + [txn])
                if res is not None:
                    return res

        memo.add(k)
        return None

    # If prev_bal is None, we can't solve deterministically; just return as-is
    if prev_bal is None:
        return (group, False)

    solved = dfs(prev_bal, 0, [])
    if solved is None:
        return (group, False)
    return (solved, True)

# -----------------------------
# Main extraction
# -----------------------------
def extract_opening_balance(pdf_path: str):
    """
    Extract OpeningBalance from statement summary text, but we DO NOT include that text in output.
    """
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            m = OPENING_BAL_RE.search(text.replace(" ", ""))
            if m:
                return to_float(m.group("bal"))
    return None

def extract_hdfc_transactions(pdf_path: str, tolerance: float = 0.50):
    # 1) Merge + tail-fix lines
    cleaned_data = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            merged_lines = merge_wrapped_lines(text)
            for txn in merged_lines:
                cleaned_data.append(pull_tail_into_narration(txn))

    # 2) Parse into txn dicts
    txns = []
    for line in cleaned_data:
        parsed = parse_txn_line(line)
        if parsed:
            txns.append(parsed)

    # 3) Opening balance (for correct debit/credit including first rows)
    opening_balance = extract_opening_balance(pdf_path)

    results = []
    prev_bal = opening_balance

    i = 0
    while i < len(txns):
        d = txns[i]["Txn_Date"]

        # collect consecutive same-date group (as it appears)
        group = [txns[i]]
        j = i + 1
        while j < len(txns) and txns[j]["Txn_Date"] == d:
            group.append(txns[j])
            j += 1

        # Try to solve correct order within this date group (fixes SWEEP/INT order issues)
        ordered_group, solved = solve_group_order(prev_bal, group, tol=tolerance)

        # Emit rows in solved order if possible, else emit in original order
        for idx, txn in enumerate(ordered_group):
            debit, credit, new_prev, ok = classify(prev_bal, txn, tol=tolerance)

            # If prev_bal is None (no opening balance found), first txn(s) can't be classified by delta.
            # To avoid blanks like your Blinkit example, fill Debit from Amount_Token ONLY for the very first emitted row.
            if prev_bal is None and len(results) == 0 and txn["Amount_Token"] is not None:
                debit = fmt_amount(abs(txn["Amount_Token"]))
                credit = ""

            results.append({
                "Date": txn["Txn_Date"],
                "Narration": txn["Narration"],
                "Ref": txn["Ref"],
                "Value_Date": txn["Value_Date"],
                "Debit": debit,
                "Credit": credit,
                "Balance": fmt_amount(txn["Closing_Balance"]),
            })
            prev_bal = new_prev

        i = j

    return results

if __name__ == "__main__":
    rows = extract_hdfc_transactions(PDF_FILE, tolerance=0.50)

    # Print in your requested format
    for r in rows:
        print(
            f"\"{r['Date']}\", "
            f"\"{r['Narration']}\", "
            f"\"{r['Ref']}\", "
            f"\"{r['Value_Date']}\", "
            f"\"{r['Debit']}\", "
            f"\"{r['Credit']}\", "
            f"\"{r['Balance']}\""
        )


"25/08/25", "UPI-KOPPULA KRISHNAVENI-BHARATPE.9007041 5461@FBPE-FDRL0001382-523756203302-PAYT OBHARATPEME", "0000523756203302", "26/08/25", "102.00", "", "32,494.95"
"26/08/25", "UPI-AMAZON INDIA-AMAZON@RAPL-RATN000RAPL -110351836863-YOUAREPAYINGFOR", "0000110351836863", "26/08/25", "604.00", "", "31,890.95"
"26/08/25", "UPI-KOPPULA KRISHNAVENI-BHARATPE.9007041 5461@FBPE-FDRL0001382-523871330581-PAYT OBHARATPEME", "0000523871330581", "26/08/25", "120.00", "", "31,770.95"
"27/08/25", "UPI-ATHMASALON SPA-Q947998856@YBL-YESB 0YBLUPI-110398935474-HAIRCUT", "0000110398935474", "27/08/25", "2,006.00", "", "29,764.95"
"27/08/25", "UPI-AMAZON INDIA-AMAZONUPI@APL-UTIB00001 00-110402977884-YOUAREPAYINGFOR", "0000110402977884", "27/08/25", "304.00", "", "29,460.95"
"27/08/25", "UPI-RAJAGOPALNAIDU PUS-9603622968@YBL- CNRB0000033-110403840327-UPI", "0000110403840327", "27/08/25", "600.00", "", "28,860.95"
"27/08/25", "UPI-SWIGGY-SWIGGYSTORES@ICICI-ICIC0DC009 9-560522913211-PAIDVIACRED", "0000560522

In [3]:
import pandas as pd

# 1. Convert the `rows` list into a Pandas DataFrame.
df_transactions = pd.DataFrame(rows)

# Convert 'Date' column to datetime objects to extract Month and Year
df_transactions['Date_dt'] = pd.to_datetime(df_transactions['Date'], format='%d/%m/%y')
df_transactions['Month'] = df_transactions['Date_dt'].dt.month
df_transactions['Year'] = df_transactions['Date_dt'].dt.year

# Change 'Date' column to 'DD-MM-YYYY' format
df_transactions['Date'] = df_transactions['Date_dt'].dt.strftime('%d-%m-%Y')

# Add 'Mode' column as empty
df_transactions['Mode'] = ''

# Add 'Purpose' column as empty
df_transactions['Purpose'] = ''

# 2. Define a list of column names with new 'Month' and 'Year' columns
# Add 'Mode' and 'Purpose' after 'Ref'
column_names = ['Date', 'Month', 'Year', 'Narration', 'Ref', 'Mode', 'Purpose', 'Debit', 'Credit', 'Balance']

# 3. Select and reorder the columns of `df_transactions`.
df_transactions = df_transactions[column_names]

# 4. Save the `df_transactions` DataFrame to an Excel file.
output_excel_path = 'HDFC_Transactions_Formatted.xlsx'
df_transactions.to_excel(output_excel_path, index=False)

print(f"DataFrame converted and saved to {output_excel_path}")
print("First 5 rows of the DataFrame:")
print(df_transactions.head())

DataFrame converted and saved to HDFC_Transactions_Formatted.xlsx
First 5 rows of the DataFrame:
         Date  Month  Year                                          Narration  \
0  25-08-2025      8  2025  UPI-KOPPULA KRISHNAVENI-BHARATPE.9007041 5461@...   
1  26-08-2025      8  2025  UPI-AMAZON INDIA-AMAZON@RAPL-RATN000RAPL -1103...   
2  26-08-2025      8  2025  UPI-KOPPULA KRISHNAVENI-BHARATPE.9007041 5461@...   
3  27-08-2025      8  2025  UPI-ATHMASALON SPA-Q947998856@YBL-YESB 0YBLUPI...   
4  27-08-2025      8  2025  UPI-AMAZON INDIA-AMAZONUPI@APL-UTIB00001 00-11...   

                Ref Mode Purpose     Debit Credit    Balance  
0  0000523756203302                 102.00         32,494.95  
1  0000110351836863                 604.00         31,890.95  
2  0000523871330581                 120.00         31,770.95  
3  0000110398935474               2,006.00         29,764.95  
4  0000110402977884                 304.00         29,460.95  


#HDFC Pixel statement format

In [ ]:
!pip install fitz

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
!pip install pymupdf
import re
import pymupdf as fitz  # PyMuPDF
import pandas as pd
from datetime import datetime
from google.colab import files

# ---------------------------------------------------
# REGEX PATTERNS
# ---------------------------------------------------

DATE_PATTERN = r"\d{1,2}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s\d{4},\s\d{2}:\d{2}"
DT_RE = re.compile(DATE_PATTERN, re.IGNORECASE)

RP_RE = re.compile(r"\+\d+")

AMT_RE = re.compile(r"(?P<sign>\+)?₹(?P<num>[0-9,]+\.\d{2})")

# ---------------------------------------------------
# DATETIME PARSER
# ---------------------------------------------------

def parse_dt(dt):
    dt = dt.replace("Sept", "Sep")
    return datetime.strptime(dt, "%d %b %Y, %H:%M")

# ---------------------------------------------------
# SPLIT LINES WITH MULTIPLE TRANSACTIONS
# ---------------------------------------------------

def split_multiple_transactions(line):

    matches = list(DT_RE.finditer(line))

    if len(matches) <= 1:
        return [line]

    parts = []
    for i, m in enumerate(matches):
        start = m.start()

        if i < len(matches)-1:
            end = matches[i+1].start()
        else:
            end = len(line)

        parts.append(line[start:end].strip())

    return parts

# ---------------------------------------------------
# CLEAN TEXT INTO PARSEABLE LINES
# ---------------------------------------------------

def normalize_lines(text):

    raw = [l.strip() for l in text.splitlines() if l.strip()]

    final_lines = []

    for l in raw:

        split_lines = split_multiple_transactions(l)

        final_lines.extend(split_lines)

    return final_lines


# ---------------------------------------------------
# TRANSACTION EXTRACTION
# ---------------------------------------------------

def extract_transactions_from_text(text):

    lines = normalize_lines(text)

    txns = []

    cur = None

    for line in lines:

        dt_match = DT_RE.search(line)

        if dt_match:

            if cur and cur["amount"] is not None:
                txns.append(cur)

            dt_str = dt_match.group()

            desc = line.replace(dt_str, "").strip()

            cur = {
                "dt": dt_str,
                "desc_parts":[desc],
                "rp":0,
                "amount":None,
                "sign":""
            }

            continue


        if cur is None:
            continue


        # reward points
        rp = RP_RE.search(line)
        if rp:
            cur["rp"] = int(rp.group().replace("+",""))
            continue


        # amount
        amt = AMT_RE.search(line)
        if amt:
            cur["amount"] = float(amt.group("num").replace(",",""))
            cur["sign"] = amt.group("sign") or ""
            continue


        # description continuation
        cur["desc_parts"].append(line)


    if cur and cur["amount"] is not None:
        txns.append(cur)


    return txns


# ---------------------------------------------------
# BUILD TRANSACTION DATAFRAME
# ---------------------------------------------------

def build_transactions_df(all_tx,file_name):

    recs = []

    for t in all_tx:

        dt = parse_dt(t["dt"])

        desc = re.sub(
            r"\s+",
            " ",
            " ".join(t["desc_parts"])
        ).strip()

        amt = t["amount"]

        debit = amt if t["sign"] != "+" else 0
        credit = amt if t["sign"] == "+" else 0

        recs.append({

            "statement_file":file_name,
            "date":dt.strftime("%d-%m-%Y"),
            "month":dt.month,
            "year":dt.year,
            "transaction_description":desc,
            "reward_point":t["rp"],
            "debit":round(debit,2),
            "credit":round(credit,2)

        })

    return pd.DataFrame(recs)


# ---------------------------------------------------
# REWARD POINT SUMMARY EXTRACTION
# ---------------------------------------------------

def extract_reward_points_earned(doc):

    for page in doc:

        words = page.get_text("words")

        for i in range(len(words)):

            if words[i][4].upper() == "EARNED":

                x = words[i][0]
                y = words[i][1]

                candidates = []

                for w in words:

                    token = w[4]

                    if token.isdigit():

                        if w[1] > y and abs(w[0]-x) < 100:

                            candidates.append((w[1],int(token)))

                if candidates:

                    candidates.sort()

                    return candidates[0][1]

    return 0


# ---------------------------------------------------
# BUILD SUMMARY
# ---------------------------------------------------

def build_statement_summary(df,earned,file_name):

    if df.empty:

        return pd.DataFrame([{
            "statement_file":file_name,
            "Date Range":"",
            "Sum of Debited Amount":"₹0",
            "Reward Points Earned":earned
        }])


    dts = pd.to_datetime(df["date"],format="%d-%m-%Y")

    start = dts.min()
    end = dts.max()

    debit_sum = df["debit"].sum()

    return pd.DataFrame([{

        "statement_file":file_name,
        "Date Range":f"{start.strftime('%d/%m/%Y')} to {end.strftime('%d/%m/%Y')}",
        "Sum of Debited Amount":f"₹{debit_sum:,.2f}",
        "Reward Points Earned":earned

    }])


# ---------------------------------------------------
# PROCESS SINGLE PDF
# ---------------------------------------------------

def extract_one_pdf(file_name):

    doc = fitz.open(file_name)

    all_tx = []

    for page in doc:

        text = page.get_text()

        tx = extract_transactions_from_text(text)

        all_tx.extend(tx)

    df_tx = build_transactions_df(all_tx,file_name)

    earned = extract_reward_points_earned(doc)

    df_summary = build_statement_summary(df_tx,earned,file_name)

    return df_tx,df_summary


# ---------------------------------------------------
# MULTI PDF EXECUTION
# ---------------------------------------------------

uploaded = files.upload()

all_txn = []
all_summary = []

for fname in uploaded.keys():

    if not fname.lower().endswith(".pdf"):
        continue

    print("Processing:",fname)

    df_txn,df_sum = extract_one_pdf(fname)

    all_txn.append(df_txn)

    all_summary.append(df_sum)


df_all_txn = pd.concat(all_txn,ignore_index=True)
df_all_summary = pd.concat(all_summary,ignore_index=True)


print("Total transactions:",len(df_all_txn))

print(df_all_summary)


# ---------------------------------------------------
# EXPORT
# ---------------------------------------------------

with pd.ExcelWriter("hdfc_pixel_multi_output.xlsx") as writer:

    df_all_txn.to_excel(writer,sheet_name="All_Transactions",index=False)

    df_all_summary.to_excel(writer,sheet_name="Statement_Summary",index=False)


print("Excel file generated.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 63.2 MB/s eta 0:00:00


Saving HDFC_PIXEL_07_2026.pdf to HDFC_PIXEL_07_2026.pdf
Saving HDFC_PIXEL_11_2025.pdf to HDFC_PIXEL_11_2025.pdf
Saving HDFC_PIXEL_03_2026.pdf to HDFC_PIXEL_03_2026.pdf
Processing: HDFC_PIXEL_07_2026.pdf
Processing: HDFC_PIXEL_11_2025.pdf
Processing: HDFC_PIXEL_03_2026.pdf
Total transactions: 7
           statement_file                Date Range Sum of Debited Amount  \
0  HDFC_PIXEL_07_2026.pdf  10/07/2026 to 10/07/2026               ₹629.96   
1  HDFC_PIXEL_11_2025.pdf  19/10/2025 to 01/11/2025             ₹1,284.94   
2  HDFC_PIXEL_03_2026.pdf  28/02/2026 to 28/02/2026               ₹200.00   

   Reward Points Earned  
0                     0  
1                     0  
2                     0  
Excel file generated.


#HDFC Credit card (new Version)

In [ ]:
# =============================
# HDFC Credit Card Parser (PyMuPDF) - WRAP SAFE + NO AMOUNT LEAK + EXCEL SAFE
# Fixes:
# 1) Wrapped description lines captured correctly (continuation logic)
# 2) "EMI" pill removed
# 3) Amount never leaks into description (removes "+ C 2,88,829.00" / "C 538.43" at end)
# 4) Debit/Credit split using amount color when available (green => Credit)
# 5) Removes illegal control chars so openpyxl won't crash (IllegalCharacterError)
# Output Excel:
# - "Transactions data"
# - "Summary"
# =============================

!pip -q install pymupdf pandas openpyxl

import re
import fitz
import pandas as pd
from datetime import datetime
from google.colab import files

OUT_XLSX = "CreditCard_Statements_Output.xlsx"

# -----------------------------
# Excel-illegal control chars (keep \t \n \r, remove other C0 controls)
# -----------------------------
ILLEGAL_XML_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def strip_illegal_xml_chars(s: str) -> str:
    if s is None:
        return ""
    return ILLEGAL_XML_RE.sub("", s)

def norm(s: str) -> str:
    s = strip_illegal_xml_chars(s or "")
    return re.sub(r"\s+", " ", s).strip()

def sanitize_df_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    df2 = df.copy()
    for c in df2.select_dtypes(include=["object", "string"]).columns:
        df2[c] = df2[c].astype(str).map(strip_illegal_xml_chars)
    return df2


# -----------------------------
# Regex
# -----------------------------
DT_LINE_RE   = re.compile(r"^(?P<dt>\d{2}/\d{2}/\d{4}\|\s*\d{2}:\d{2})$")                 # dt alone
DT_INLINE_RE = re.compile(r"^(?P<dt>\d{2}/\d{2}/\d{4}\|\s*\d{2}:\d{2})\s+(?P<rest>.+)$")  # dt + something

AMOUNT_LINE_RE = re.compile(r"^\s*C\s*(?P<num>\d{1,3}(?:,\d{2,3})*\.\d{2})\s*$")          # "C 538.43"
REWARD_LINE_RE = re.compile(r"^\s*([+-])\s*(\d{1,6})\s*$")                                # "+ 388" or "- 40"
PAGE_RE        = re.compile(r"^Page\s+\d+\s+of\s+\d+$", re.IGNORECASE)

# Remove amount that sometimes appears inside description lines:
# "... + C 2,88,829.00" or "... C 538.43"
TRAILING_AMT_IN_DESC_RE = re.compile(r"\s*(?:\+\s*)?C\s*\d{1,3}(?:,\d{2,3})*\.\d{2}\s*$")

DOMESTIC_SECTION_RE = re.compile(r"^Domestic\s+Transactions?$", re.IGNORECASE)
STOP_SECTION_RE = re.compile(
    r"^(Past Dues|International Transactions|Rewards Program|Eligible for EMI|Page\s+\d+\s+of\s+\d+)\b",
    re.IGNORECASE,
)

BAD_TAGS = {"EMI"}  # remove from description anywhere

# For summary
BILLING_RE = re.compile(
    r"Billing\s+Period\s*\n\s*(\d{1,2}\s+[A-Za-z]{3,9},?\s+\d{4})\s*[-–]\s*(\d{1,2}\s+[A-Za-z]{3,9},?\s+\d{4})",
    re.IGNORECASE,
)
PTS_TOTAL_RE = re.compile(r"\bTotal\s+(\d+)\s*pts\b", re.IGNORECASE)


# -----------------------------
# Helpers
# -----------------------------
def parse_dt(dt_text: str):
    dt_text = strip_illegal_xml_chars(dt_text).replace(" ", "")
    d, t = dt_text.split("|")
    return d, t

def parse_amount_num(s: str):
    return float(s.replace(",", ""))

def int_to_rgb(color_int: int):
    r = (color_int >> 16) & 255
    g = (color_int >> 8) & 255
    b = color_int & 255
    return r, g, b

def is_greenish(color_int: int) -> bool:
    r, g, b = int_to_rgb(color_int)
    return (g >= 110) and (g > r + 25) and (g > b + 25)

def parse_ddmonyyyy(s: str):
    s = strip_illegal_xml_chars(s or "")
    for fmt in ("%d %b, %Y", "%d %b %Y", "%d %B, %Y", "%d %B %Y"):
        try:
            return datetime.strptime(re.sub(r"\s+", " ", s).strip(), fmt)
        except ValueError:
            pass
    return None

def extract_full_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    text = "\n".join([(p.get_text("text") or "") for p in doc])
    return strip_illegal_xml_chars(text)


# -----------------------------
# Span helpers (for credit/debit via green amount)
# -----------------------------
def extract_spans(page: fitz.Page):
    d = page.get_text("dict")
    spans = []
    for block in d.get("blocks", []):
        for line in block.get("lines", []):
            for sp in line.get("spans", []):
                txt = norm(sp.get("text", ""))
                if not txt:
                    continue
                x0, y0, x1, y1 = sp["bbox"]
                spans.append({
                    "text": txt,
                    "x0": float(x0),
                    "y0": float(y0),
                    "x1": float(x1),
                    "y1": float(y1),
                    "color": int(sp.get("color", 0)),
                })
    spans.sort(key=lambda z: (round(z["y0"], 2), z["x0"]))
    return spans

def find_dt_y(spans, dt_text):
    key = strip_illegal_xml_chars(dt_text).replace(" ", "")
    ys = [sp["y0"] for sp in spans if sp["text"].replace(" ", "") == key]
    return min(ys) if ys else None

def pick_amount_color_in_band(spans, y_start, y_end):
    """
    Pick right-most numeric amount-like span in amount area in the band.
    Returns (amount_float_or_none, color_int)
    """
    X_AMT_MIN = 420
    candidates = []
    for sp in spans:
        if not (y_start <= sp["y0"] < y_end):
            continue
        if sp["x0"] < X_AMT_MIN:
            continue

        txt = sp["text"].replace("₹", "").strip()
        txt = txt.replace("C", "").strip()
        if re.fullmatch(r"\d{1,3}(?:,\d{2,3})*\.\d{2}", txt):
            candidates.append((sp["x0"], parse_amount_num(txt), sp["color"]))

    if not candidates:
        return None, 0

    candidates.sort(key=lambda x: x[0], reverse=True)
    _, amt, col = candidates[0]
    return amt, col


# -----------------------------
# Parse one page using TEXT lines (wrap-safe)
# -----------------------------
def parse_domestic_transactions_from_page(page: fitz.Page):
    raw = page.get_text("text") or ""
    raw = strip_illegal_xml_chars(raw)

    text_lines = [norm(x) for x in raw.splitlines()]
    text_lines = [x for x in text_lines if x]

    # Find start of Domestic Transactions
    start_idx = None
    for i, ln in enumerate(text_lines):
        if DOMESTIC_SECTION_RE.match(ln):
            start_idx = i
            break
    if start_idx is None:
        return []

    rows = []
    cur = None  # {"dt":..., "desc_lines":[...], "rewards":..., "amount_line":...}

    def flush():
        nonlocal cur
        if cur:
            rows.append(cur)
            cur = None

    i = start_idx + 1
    while i < len(text_lines):
        ln = text_lines[i]

        if STOP_SECTION_RE.match(ln):
            flush()
            break

        # headers may be split across multiple lines; just skip each token line
        if ln.upper() in {"DATE & TIME", "TRANSACTION DESCRIPTION", "REWARDS", "AMOUNT", "PI"}:
            i += 1
            continue

        # Skip client name line before first txn (e.g., CHRISTOPHER)
        if cur is None and re.fullmatch(r"[A-Z ]{3,40}", ln):
            i += 1
            continue

        m_inline = DT_INLINE_RE.match(ln)
        m_dt = DT_LINE_RE.match(ln)

        if m_inline:
            flush()
            cur = {"dt": m_inline.group("dt"), "desc_lines": [m_inline.group("rest")], "rewards": None, "amount_line": None}
            i += 1
            continue

        if m_dt:
            flush()
            cur = {"dt": m_dt.group("dt"), "desc_lines": [], "rewards": None, "amount_line": None}
            i += 1
            continue

        # Continuation lines
        if cur:
            if ln.upper() in BAD_TAGS:
                i += 1
                continue

            m_r = REWARD_LINE_RE.match(ln)
            if m_r:
                sign = 1 if m_r.group(1) == "+" else -1
                cur["rewards"] = sign * int(m_r.group(2))
                i += 1
                continue

            m_a = AMOUNT_LINE_RE.match(ln)
            if m_a:
                cur["amount_line"] = ln
                i += 1
                continue

            if ln == "l":  # PI marker sometimes isolated
                i += 1
                continue

            if PAGE_RE.match(ln):
                flush()
                break

            cur["desc_lines"].append(ln)

        i += 1

    flush()
    return rows


# -----------------------------
# Convert parsed rows -> records
# -----------------------------
def rows_to_records(page: fitz.Page, rows):
    spans = extract_spans(page)
    records = []

    for r in rows:
        dt_text = r["dt"]
        date_str, time_str = parse_dt(dt_text)

        # ---- Description: join wrapped lines, stripping any trailing "+ C amount" from each line
        desc_parts = []
        for x in r["desc_lines"]:
            x = norm(x)
            if not x:
                continue
            if x.upper() in BAD_TAGS:
                continue

            # ✅ remove amount if it appears at end of the description line
            x = TRAILING_AMT_IN_DESC_RE.sub("", x).strip()
            if not x:
                continue

            desc_parts.append(x)

        description = norm(" ".join(desc_parts)) or None

        # ✅ final safety
        if description:
            description = TRAILING_AMT_IN_DESC_RE.sub("", description).strip() or None

        # ---- Amount numeric: prefer amount_line, else span numeric
        amt = None
        if r.get("amount_line"):
            m_a = AMOUNT_LINE_RE.match(r["amount_line"])
            if m_a:
                amt = parse_amount_num(m_a.group("num"))

        # ---- Determine band for color-based credit/debit
        dt_y = find_dt_y(spans, dt_text)
        amt_color = 0
        if dt_y is not None:
            next_dt_ys = []
            for sp in spans:
                t = sp["text"].replace(" ", "")
                if re.fullmatch(r"\d{2}/\d{2}/\d{4}\|\d{2}:\d{2}", t):
                    if sp["y0"] > dt_y + 0.5:
                        next_dt_ys.append(sp["y0"])
            y_end = min(next_dt_ys) if next_dt_ys else (dt_y + 70)

            amt2, col2 = pick_amount_color_in_band(spans, dt_y - 1.0, y_end)
            if amt is None and amt2 is not None:
                amt = amt2
            amt_color = col2

        debit = credit = None
        if amt is not None:
            if amt_color and is_greenish(amt_color):
                credit = amt
            else:
                debit = amt

        # ---- Date breakdown
        try:
            dt_obj = datetime.strptime(date_str, "%d/%m/%Y")
            date_out = dt_obj.strftime("%d-%m-%Y")
            month = dt_obj.strftime("%m")
            year = dt_obj.year
        except ValueError:
            date_out, month, year = date_str, None, None

        records.append({
            "Date": date_out,
            "Month": month,
            "Year": year,
            "Transaction_Details": description,
            "Reward_Points": r.get("rewards"),
            "Debit": debit,
            "Credit": credit,
            "Balance": None,
        })

    return records


def extract_transactions_for_pdf(pdf_path: str) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    all_records = []
    for page in doc:
        rows = parse_domestic_transactions_from_page(page)
        all_records.extend(rows_to_records(page, rows))
    return pd.DataFrame(all_records)


# -----------------------------
# Summary helpers
# -----------------------------
def extract_billing_period(full_text: str):
    m = BILLING_RE.search(full_text or "")
    if not m:
        return None
    d1 = parse_ddmonyyyy(m.group(1))
    d2 = parse_ddmonyyyy(m.group(2))
    if not d1 or not d2:
        return None
    return f"{d1.strftime('%d/%m/%Y')} to {d2.strftime('%d/%m/%Y')}"

def derive_date_range_and_debit_sum(df_txn: pd.DataFrame, full_text: str):
    billing = extract_billing_period(full_text)
    if billing:
        debit_sum = round(float(df_txn["Debit"].fillna(0).sum()), 2) if not df_txn.empty else 0.0
        return billing, debit_sum

    if df_txn.empty:
        return None, 0.0

    # fallback from min/max dates present
    df_valid = df_txn[df_txn["Date"].notna()].copy()
    if df_valid.empty:
        return None, 0.0

    # parse dd-mm-yyyy back to dt safely
    parsed = []
    for s in df_valid["Date"].astype(str):
        try:
            parsed.append(datetime.strptime(s, "%d-%m-%Y"))
        except ValueError:
            pass
    if not parsed:
        return None, round(float(df_txn["Debit"].fillna(0).sum()), 2)

    start_dt = min(parsed)
    end_dt = max(parsed)
    date_range = f"{start_dt.strftime('%d/%m/%Y')} to {end_dt.strftime('%d/%m/%Y')}"
    debit_sum = round(float(df_txn["Debit"].fillna(0).sum()), 2)
    return date_range, debit_sum

def extract_points_earned(full_text: str):
    m = PTS_TOTAL_RE.search(full_text or "")
    return int(m.group(1)) if m else None


# -----------------------------
# Process multiple PDFs + save Excel
# -----------------------------
def process_multiple_pdfs(pdf_paths: list[str], out_xlsx: str):
    all_txn = []
    summary_rows = []

    for pdf_path in pdf_paths:
        file_name = pdf_path.split("/")[-1]
        full_text = extract_full_text(pdf_path)

        df_txn = extract_transactions_for_pdf(pdf_path)
        date_range, debit_sum = derive_date_range_and_debit_sum(df_txn, full_text)
        points_earned = extract_points_earned(full_text)

        if df_txn.empty:
            summary_rows.append({
                "File_Name": file_name,
                "Date Range": date_range,
                "Sum of Debit Amount": debit_sum,
                "Reward Points Earned": points_earned,
                "Status": "No transactions extracted",
            })
            continue

        df_txn.insert(0, "File_Name", file_name)
        all_txn.append(df_txn)

        summary_rows.append({
            "File_Name": file_name,
            "Date Range": date_range,
            "Sum of Debit Amount": debit_sum,
            "Reward Points Earned": points_earned,
            "Status": "OK",
        })

    df_all = pd.concat(all_txn, ignore_index=True) if all_txn else pd.DataFrame(
        columns=["File_Name","Date","Month","Year","Transaction_Details","Reward_Points","Debit","Credit","Balance"]
    )
    df_sum = pd.DataFrame(summary_rows)

    # ✅ CRITICAL: sanitize before writing to Excel
    df_all = sanitize_df_for_excel(df_all)
    df_sum = sanitize_df_for_excel(df_sum)

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        df_all.to_excel(writer, sheet_name="Transactions data", index=False)
        df_sum.to_excel(writer, sheet_name="Summary", index=False)

    return df_all, df_sum


# -----------------------------
# Run (Colab)
# -----------------------------
uploaded = files.upload()
pdf_paths = [f"/content/{name}" for name in uploaded.keys()]

df_all, df_sum = process_multiple_pdfs(pdf_paths, OUT_XLSX)

print("✅ Saved:", OUT_XLSX)
print(df_sum)

files.download(OUT_XLSX)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.7 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Saving Aug2026_Billedstatements_5082_12-10-26_22-22.pdf to Aug2026_Billedstatements_5082_12-10-26_22-22.pdf
Saving Apr2026_Billedstatements_5082_12-10-26_22-22.pdf to Apr2026_Billedstatements_5082_12-10-26_22-22.pdf
Saving Jul2026_Billedstatements_5082_12-10-26_22-22.pdf to Jul2026_Billedstatements_5082_12-10-26_22-22.pdf
Saving Jun2026_Billedstatements_5082_12-10-26_22-22.pdf to Jun2026_Billedstatements_5082_12-10-26_22-22.pdf
Saving Mar2026_Billedstatements_5082_12-10-26_22-23.pdf to Mar2026_Billedstatements_5082_12-10-26_22-23.pdf
Saving May2026_Billedstatements_5082_12-10-26_22-22.pdf to May2026_Billedstatements_5082_12-10-26_22-22.pdf
Saving Sep2026_Billedstatements_5082_12-10-26_22-22.pdf to Sep2026_Billedstatements_5082_12-10-26_22-22.pdf


/tmp/ipykernel_2274/558473351.py:444: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(all_txn, ignore_index=True) if all_txn else pd.DataFrame(


✅ Saved: CreditCard_Statements_Output.xlsx
                                          File_Name                Date Range  \
0  Aug2026_Billedstatements_5082_12-10-26_22-22.pdf  15/07/2026 to 14/08/2026   
1  Apr2026_Billedstatements_5082_12-10-26_22-22.pdf  16/03/2026 to 14/04/2026   
2  Jul2026_Billedstatements_5082_12-10-26_22-22.pdf  17/06/2026 to 13/07/2026   
3  Jun2026_Billedstatements_5082_12-10-26_22-22.pdf  18/05/2026 to 12/06/2026   
4  Mar2026_Billedstatements_5082_12-10-26_22-23.pdf  08/03/2026 to 14/03/2026   
5  May2026_Billedstatements_5082_12-10-26_22-22.pdf  17/04/2026 to 14/05/2026   
6  Sep2026_Billedstatements_5082_12-10-26_22-22.pdf  19/08/2026 to 15/09/2026   

   Sum of Debit Amount  Reward Points Earned Status  
0             18321.57                   356     OK  
1             80069.11                  1316     OK  
2             26131.49                   107     OK  
3            191149.52                   564     OK  
4             14827.00                

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 🏦 HDFC Bank Credit Card Statement Extractor (v2 — Fixed)
**Extracts ALL transaction data from HDFC Bank CC Year-End Statement PDF → Structured Excel**

### Fixes over v1:
- Uses `pdfplumber` for accurate table-based extraction (PyPDF2 garbles table text)
- Multi-strategy pipeline: Table extraction → Text regex → Line-merging fallback
- Greedy regex with lookahead to avoid description/amount confusion
- Line-merging logic for transactions split across multiple lines
- Deduplication to prevent double-counting
- Validation count shown at end

---
**Run each cell in order ↓**

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q pdfplumber openpyxl pandas PyPDF2 tabula-py
print('✅ All dependencies installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 56.5 MB/s eta 0:00:00
✅ All dependencies installed!


In [ ]:
# ============================================================
# CELL 2: Import Libraries & Upload PDF
# ============================================================
import pandas as pd
import re
from datetime import datetime
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

import pdfplumber

try:
    from PyPDF2 import PdfReader
    PYPDF2_AVAILABLE = True
except ImportError:
    PYPDF2_AVAILABLE = False

try:
    import tabula
    TABULA_AVAILABLE = True
except ImportError:
    TABULA_AVAILABLE = False

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('✅ Libraries loaded!')
print('\n📂 Upload your HDFC Bank Credit Card Statement PDF:')
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {pdf_filename}')

✅ Libraries loaded!

📂 Upload your HDFC Bank Credit Card Statement PDF:


Saving 04-2024 to 03 2025 - Deepanshu.pdf to 04-2024 to 03 2025 - Deepanshu.pdf

✅ Uploaded: 04-2024 to 03 2025 - Deepanshu.pdf


In [ ]:
# ============================================================
# CELL 3: Dynamic Extraction Engine (FIXED)
# ============================================================

class HDFCCCStatementExtractor:
    """
    Robust, dynamic extractor for HDFC Bank Credit Card Year-End Statements.
    Uses a multi-strategy pipeline to ensure ALL transactions are captured.

    Strategy Order:
      1. pdfplumber table extraction (best for HDFC table-based PDFs)
      2. pdfplumber raw text + regex (fallback if tables fail)
      3. PyPDF2 text + regex (secondary fallback)
      4. tabula-py table extraction (last resort)
    """

    # Date pattern to identify transaction lines
    DATE_PATTERN = re.compile(r'\d{2}-[A-Za-z]{3}-\d{4}')

    # FIXED regex: uses GREEDY .+ with a lookahead to anchor the amount
    # This avoids the non-greedy (.+?) bug that could grab wrong amounts
    # Lookahead: amount must be followed by whitespace and DR/CR
    TRANSACTION_REGEX = re.compile(
        r'(\d{2}-[A-Za-z]{3}-\d{4})'
        r'\s+'
        r'(.+?)'
        r'\s+'
        r'([\d,]+\.\d{2})'
        r'\s+'
        r'(DR|CR)'
        r'(?:\s+\d{4,}X+\d+)?',   # optional card number
        re.IGNORECASE
    )

    # Stricter pattern: amount followed by DR/CR at end of meaningful content
    TRANSACTION_REGEX_STRICT = re.compile(
        r'(\d{2}-[A-Za-z]{3}-\d{4})'
        r'\s+'
        r'(.+)'
        r'\s+'
        r'([\d,]+\.\d{2})'
        r'\s+'
        r'(DR|CR)'
        r'\s*$',
        re.IGNORECASE
    )

    # Lines to skip (headers, footers)
    SKIP_PATTERNS = [
        r'Transaction\s*Details',
        r'Date\s+Transaction\s+Description',
        r'Year\s*End\s*Statement',
        r'HDFC\s*Bank\s*Credit',
        r'Page\s+\d+\s+of\s+\d+',
        r'Account\s*Summary',
        r'Card\s*Number\s+Number',
        r'Monthly\s*Statement',
        r'Primary\s*Card\s*Holder',
        r'Month\s+Card\s+Number',
        r'Debits\s+Minimum',
        r'Domestic.*International.*Cash',
        r'HSN\s*Code',
        r'^GSTIN',
        r'^\s*$',
        r'Number\s+of\s+Add-on',
        r'Purchases\s*&\s*Debits',
        r'Payments\s*&\s*Credits',
        r'Credit\s+Limit\s+Monthly',
        r'^\d{12,}X+\d+\s+\d+\s+',  # Account summary row
        r'^Amount\s+DR/CR',
    ]

    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.transactions = []
        self.card_number = None
        self.credit_limit = None
        self.statement_period = None
        self._seen_keys = set()  # For deduplication

    def extract(self):
        """Main extraction pipeline with multiple strategies."""
        print('=' * 65)
        print('  HDFC CC Statement Extraction — Multi-Strategy Pipeline')
        print('=' * 65)

        # Strategy 1: pdfplumber table extraction (BEST for HDFC)
        print('\n▶ Strategy 1: pdfplumber TABLE extraction')
        self._extract_pdfplumber_tables()
        count_after_s1 = len(self.transactions)
        print(f'  → Found {count_after_s1} transactions')

        # Strategy 2: pdfplumber raw text + regex
        print('\n▶ Strategy 2: pdfplumber TEXT + regex')
        self._extract_pdfplumber_text()
        count_after_s2 = len(self.transactions)
        print(f'  → Found {count_after_s2 - count_after_s1} new (total: {count_after_s2})')

        # Strategy 3: PyPDF2 text + regex (may catch different text)
        if PYPDF2_AVAILABLE:
            print('\n▶ Strategy 3: PyPDF2 TEXT + regex')
            self._extract_pypdf2_text()
            count_after_s3 = len(self.transactions)
            print(f'  → Found {count_after_s3 - count_after_s2} new (total: {count_after_s3})')

        # Strategy 4: tabula-py (last resort)
        if TABULA_AVAILABLE and len(self.transactions) < 10:
            print('\n▶ Strategy 4: tabula-py TABLE extraction')
            self._extract_tabula()
            print(f'  → Total after tabula: {len(self.transactions)}')

        if not self.transactions:
            print('\n❌ No transactions extracted. Check PDF format.')
            return pd.DataFrame()

        print(f'\n{"="*65}')
        print(f'  ✅ TOTAL UNIQUE TRANSACTIONS EXTRACTED: {len(self.transactions)}')
        print(f'{"="*65}')
        return self._build_dataframe()

    # ------------------------------------------------------------------
    # Strategy 1: pdfplumber TABLE extraction
    # ------------------------------------------------------------------
    def _extract_pdfplumber_tables(self):
        """Extract using pdfplumber's table detection — best for HDFC PDFs."""
        try:
            with pdfplumber.open(self.pdf_path) as pdf:
                # Extract metadata from first page
                if pdf.pages:
                    first_text = pdf.pages[0].extract_text() or ''
                    self._extract_metadata(first_text)

                for page_num, page in enumerate(pdf.pages):
                    tables = page.extract_tables()
                    for table in tables:
                        if not table:
                            continue
                        for row in table:
                            if not row:
                                continue
                            # Clean None values
                            cells = [str(c).strip() if c else '' for c in row]
                            self._parse_table_row(cells, page_num + 1)
        except Exception as e:
            print(f'  ⚠ pdfplumber table extraction error: {e}')

    def _parse_table_row(self, cells, page_num):
        """Parse a table row — handles both structured columns and merged text."""
        # Approach A: Structured columns (Date | Description | Amount | DR/CR | Card)
        if len(cells) >= 4:
            date_cell = cells[0].strip()
            if self.DATE_PATTERN.match(date_cell):
                # Find amount and DR/CR in remaining cells
                description = ''
                amount_str = ''
                dr_cr = ''

                for i in range(1, len(cells)):
                    cell = cells[i].strip()
                    if cell.upper() in ('DR', 'CR'):
                        dr_cr = cell.upper()
                    elif re.match(r'^[\d,]+\.\d{2}$', cell):
                        amount_str = cell
                    elif cell and not re.match(r'^\d{4,}X+\d+$', cell):
                        # Not a card number
                        if not description:
                            description = cell
                        else:
                            description += ' ' + cell

                if date_cell and description and amount_str and dr_cr:
                    self._add_transaction(date_cell, description, amount_str, dr_cr)
                    return

        # Approach B: Merged text in single/few cells — use regex
        merged = ' '.join(cells)
        self._parse_line_with_regex(merged)

    # ------------------------------------------------------------------
    # Strategy 2: pdfplumber TEXT extraction + regex
    # ------------------------------------------------------------------
    def _extract_pdfplumber_text(self):
        """Extract using pdfplumber raw text (catches rows missed by tables)."""
        try:
            all_lines = []
            with pdfplumber.open(self.pdf_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        all_lines.extend(text.split('\n'))

            # Also try line-merging for split transactions
            merged_lines = self._merge_split_lines(all_lines)
            for line in merged_lines:
                self._parse_line_with_regex(line)
        except Exception as e:
            print(f'  ⚠ pdfplumber text extraction error: {e}')

    # ------------------------------------------------------------------
    # Strategy 3: PyPDF2 text + regex
    # ------------------------------------------------------------------
    def _extract_pypdf2_text(self):
        """Fallback using PyPDF2 (different text extraction algorithm)."""
        try:
            reader = PdfReader(self.pdf_path)
            all_lines = []
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    all_lines.extend(text.split('\n'))

            merged_lines = self._merge_split_lines(all_lines)
            for line in merged_lines:
                self._parse_line_with_regex(line)
        except Exception as e:
            print(f'  ⚠ PyPDF2 extraction error: {e}')

    # ------------------------------------------------------------------
    # Strategy 4: tabula-py
    # ------------------------------------------------------------------
    def _extract_tabula(self):
        """Last resort: tabula-py table extraction."""
        try:
            for mode in ['lattice', 'stream']:
                dfs = tabula.read_pdf(
                    self.pdf_path, pages='all', multiple_tables=True,
                    lattice=(mode == 'lattice'), stream=(mode == 'stream'),
                    pandas_options={'header': None}
                )
                for df in (dfs or []):
                    for _, row in df.iterrows():
                        row_text = ' '.join(str(v) for v in row.values if pd.notna(v))
                        self._parse_line_with_regex(row_text)
        except Exception as e:
            print(f'  ⚠ Tabula extraction error: {e}')

    # ------------------------------------------------------------------
    # Core parsing helpers
    # ------------------------------------------------------------------
    def _parse_line_with_regex(self, line):
        """Parse a single line using regex patterns."""
        line = line.strip()
        if not line or len(line) < 15:
            return

        # Check skip patterns
        for pattern in self.SKIP_PATTERNS:
            if re.search(pattern, line, re.IGNORECASE):
                return

        # Must start with or contain a date
        if not self.DATE_PATTERN.search(line):
            return

        # Try strict pattern first (amount + DR/CR at end of line)
        match = self.TRANSACTION_REGEX_STRICT.search(line)
        if match:
            self._add_transaction(
                match.group(1), match.group(2).strip(),
                match.group(3), match.group(4).upper()
            )
            return

        # Try standard pattern
        match = self.TRANSACTION_REGEX.search(line)
        if match:
            self._add_transaction(
                match.group(1), match.group(2).strip(),
                match.group(3), match.group(4).upper()
            )

    def _merge_split_lines(self, lines):
        """
        Merge lines where a transaction is split across 2+ lines.
        If line N has a date but no DR/CR, and line N+1 has no date
        but has DR/CR, merge them.
        """
        merged = []
        i = 0
        while i < len(lines):
            line = lines[i].strip()

            # Check if line has a date but no DR/CR (possibly split)
            has_date = bool(self.DATE_PATTERN.match(line))
            has_drcr = bool(re.search(r'\b(DR|CR)\b', line))

            if has_date and not has_drcr and (i + 1) < len(lines):
                next_line = lines[i + 1].strip()
                next_has_date = bool(self.DATE_PATTERN.match(next_line))
                next_has_drcr = bool(re.search(r'\b(DR|CR)\b', next_line))

                if not next_has_date and next_has_drcr:
                    merged.append(line + ' ' + next_line)
                    i += 2
                    continue
                # Also check 2 lines ahead
                elif not next_has_date and (i + 2) < len(lines):
                    next2 = lines[i + 2].strip()
                    if bool(re.search(r'\b(DR|CR)\b', next2)):
                        merged.append(line + ' ' + next_line + ' ' + next2)
                        i += 3
                        continue

            merged.append(line)
            i += 1

        return merged

    def _add_transaction(self, date_str, description, amount_str, dr_cr):
        """Add transaction with deduplication."""
        try:
            parsed_date = datetime.strptime(date_str.strip(), '%d-%b-%Y')
            amount = float(amount_str.replace(',', ''))

            # Clean description
            description = re.sub(r'\s*\d{4,}X+\d+\s*', '', description).strip()
            description = re.sub(r'\s{2,}', ' ', description).strip()

            # Deduplication key: date + amount + type + first 20 chars of desc
            dedup_key = (
                parsed_date.strftime('%Y%m%d'),
                f'{amount:.2f}',
                dr_cr,
                description[:20].lower()
            )

            if dedup_key in self._seen_keys:
                return  # Skip duplicate
            self._seen_keys.add(dedup_key)

            self.transactions.append({
                'date': parsed_date,
                'description': description,
                'amount': amount,
                'dr_cr': dr_cr
            })
        except (ValueError, AttributeError):
            pass

    def _extract_metadata(self, text):
        """Extract card number, credit limit, statement period."""
        card_match = re.search(r'(\d{3,6}\d*X+\d+)', text)
        if card_match:
            self.card_number = card_match.group(1)

        limit_match = re.search(r'Credit\s*Limit[\s:]*([\d,]+\.?\d*)', text, re.IGNORECASE)
        if limit_match:
            self.credit_limit = limit_match.group(1)

        period_match = re.search(r'period\s+from\s+(\S+)\s+to\s+(\S+)', text, re.IGNORECASE)
        if period_match:
            self.statement_period = f'{period_match.group(1)} to {period_match.group(2)}'

        if self.card_number:
            print(f'  Card: {self.card_number}')
        if self.credit_limit:
            print(f'  Credit Limit: ₹{self.credit_limit}')
        if self.statement_period:
            print(f'  Period: {self.statement_period}')

    def _build_dataframe(self):
        """Build the final structured DataFrame."""
        records = []
        for txn in self.transactions:
            dt = txn['date']
            records.append({
                'Date (DD-MM-YYYY)': dt.strftime('%d-%m-%Y'),
                'Month': dt.month,
                'Year': dt.year,
                'Description': txn['description'],
                'Cheque Reference No.': '',
                'Mode': '',
                'Purpose': '',
                'Debit': txn['amount'] if txn['dr_cr'] == 'DR' else None,
                'Credit': txn['amount'] if txn['dr_cr'] == 'CR' else None,
                'Balance': ''
            })

        df = pd.DataFrame(records)
        df['_sort'] = pd.to_datetime(df['Date (DD-MM-YYYY)'], format='%d-%m-%Y')
        df = df.sort_values('_sort').drop(columns=['_sort']).reset_index(drop=True)
        return df


print('✅ Extraction engine loaded!')

✅ Extraction engine loaded!


In [ ]:
# ============================================================
# CELL 4: Professional Excel Writer
# ============================================================

def save_to_excel(df, output_filename, extractor):
    """Save DataFrame to professionally formatted Excel."""
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Transactions'

    # Styles
    hdr_font = Font(name='Arial', bold=True, size=11, color='FFFFFF')
    hdr_fill = PatternFill('solid', fgColor='1F4E79')
    hdr_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
    data_font = Font(name='Arial', size=10)
    dr_font = Font(name='Arial', size=10, color='CC0000')
    cr_font = Font(name='Arial', size=10, color='006600')
    thin_border = Border(
        left=Side('thin', color='D9D9D9'), right=Side('thin', color='D9D9D9'),
        top=Side('thin', color='D9D9D9'), bottom=Side('thin', color='D9D9D9')
    )
    alt_fill = PatternFill('solid', fgColor='F2F7FB')

    # Title row
    ws.merge_cells('A1:J1')
    ws['A1'].value = 'HDFC Bank Credit Card Statement — Transaction Details'
    ws['A1'].font = Font(name='Arial', bold=True, size=14, color='1F4E79')
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 30

    # Metadata row
    ws.merge_cells('A2:J2')
    meta = []
    if extractor.card_number:
        meta.append(f'Card: {extractor.card_number}')
    if extractor.statement_period:
        meta.append(f'Period: {extractor.statement_period}')
    meta.append(f'Transactions: {len(df)}')
    ws['A2'].value = ' | '.join(meta)
    ws['A2'].font = Font(name='Arial', size=10, italic=True, color='666666')
    ws['A2'].alignment = Alignment(horizontal='center')

    # Headers at row 4
    headers = list(df.columns)
    for c, h in enumerate(headers, 1):
        cell = ws.cell(row=4, column=c, value=h)
        cell.font = hdr_font
        cell.fill = hdr_fill
        cell.alignment = hdr_align
        cell.border = thin_border
    ws.row_dimensions[4].height = 28

    # Data rows starting at row 5
    for r_idx, (_, row) in enumerate(df.iterrows(), 5):
        for c_idx, h in enumerate(headers, 1):
            val = row[h]
            cell = ws.cell(row=r_idx, column=c_idx)

            if pd.isna(val) or val == '':
                cell.value = None
            elif h in ('Debit', 'Credit'):
                cell.value = val
                cell.number_format = '#,##0.00'
                cell.font = dr_font if h == 'Debit' else cr_font
            elif h in ('Month', 'Year'):
                cell.value = int(val) if pd.notna(val) else None
                cell.font = data_font
            else:
                cell.value = str(val)
                cell.font = data_font

            cell.border = thin_border
            cell.alignment = Alignment(
                horizontal='left' if h == 'Description' else 'center',
                vertical='center'
            )
            if (r_idx - 5) % 2 == 1:
                cell.fill = alt_fill

    # Column widths
    widths = {
        'Date (DD-MM-YYYY)': 18, 'Month': 8, 'Year': 8, 'Description': 48,
        'Cheque Reference No.': 22, 'Mode': 10, 'Purpose': 12,
        'Debit': 15, 'Credit': 15, 'Balance': 15
    }
    for c, h in enumerate(headers, 1):
        ws.column_dimensions[get_column_letter(c)].width = widths.get(h, 15)

    # Summary section
    last_row = len(df) + 4
    sr = last_row + 2
    ws.cell(row=sr, column=1, value='SUMMARY').font = Font(name='Arial', bold=True, size=12, color='1F4E79')

    summary_formulas = [
        ('Total Debits (₹)', f'=SUM(H5:H{last_row})', '#,##0.00'),
        ('Total Credits (₹)', f'=SUM(I5:I{last_row})', '#,##0.00'),
        ('Net Spend (₹)', f'=SUM(H5:H{last_row})-SUM(I5:I{last_row})', '#,##0.00'),
        ('Total Debit Transactions', f'=COUNTA(H5:H{last_row})', '0'),
        ('Total Credit Transactions', f'=COUNTA(I5:I{last_row})', '0'),
        ('Total Transactions', f'=COUNTA(A5:A{last_row})', '0'),
    ]
    for i, (label, formula, fmt) in enumerate(summary_formulas):
        r = sr + 1 + i
        ws.cell(row=r, column=1, value=label).font = Font(name='Arial', bold=True, size=10)
        vc = ws.cell(row=r, column=2, value=formula)
        vc.font = Font(name='Arial', size=10)
        vc.number_format = fmt

    ws.freeze_panes = 'A5'
    ws.auto_filter.ref = f'A4:J{last_row}'

    wb.save(output_filename)
    print(f'\n💾 Saved: {output_filename}')


print('✅ Excel writer loaded!')

✅ Excel writer loaded!


In [ ]:
# ============================================================
# CELL 5: Run Extraction, Validate & Download
# ============================================================

# Extract
extractor = HDFCCCStatementExtractor(pdf_filename)
df = extractor.extract()

if not df.empty:
    # Preview
    print('\n' + '=' * 65)
    print('  PREVIEW (First 15 rows):')
    print('=' * 65)
    display(df.head(15))

    # Validation summary
    total_dr = df['Debit'].sum()
    total_cr = df['Credit'].sum()
    dr_count = df['Debit'].notna().sum()
    cr_count = df['Credit'].notna().sum()

    print(f'\n📊 Debit transactions:  {dr_count}  |  Total: ₹{total_dr:,.2f}')
    print(f'📊 Credit transactions: {cr_count}  |  Total: ₹{total_cr:,.2f}')
    print(f'📊 Net spend:           ₹{total_dr - total_cr:,.2f}')
    print(f'📊 Grand total transactions: {len(df)}')

    # Month-wise breakdown
    print('\n📅 Month-wise transaction count:')
    monthly = df.groupby(['Year', 'Month']).size().reset_index(name='Count')
    for _, row in monthly.iterrows():
        print(f'   {int(row["Year"])}-{int(row["Month"]):02d}: {row["Count"]} transactions')

    # Save & download
    output_xlsx = pdf_filename.replace('.pdf', '').replace('.PDF', '') + '_Transactions.xlsx'
    save_to_excel(df, output_xlsx, extractor)

    print('\n📥 Downloading...')
    files.download(output_xlsx)
    print('\n✅ Done! Check your downloads.')
else:
    print('\n❌ No transactions found.')

  HDFC CC Statement Extraction — Multi-Strategy Pipeline

▶ Strategy 1: pdfplumber TABLE extraction
  Card: 000558983XXXXXX1994
  Credit Limit: ₹89
  Period: APRIL-24 to MARCH-25
  → Found 89 transactions

▶ Strategy 2: pdfplumber TEXT + regex
  → Found 0 new (total: 89)

▶ Strategy 3: PyPDF2 TEXT + regex
  → Found 0 new (total: 89)

  ✅ TOTAL UNIQUE TRANSACTIONS EXTRACTED: 89

  PREVIEW (First 15 rows):


,Date (DD-MM-YYYY),Month,Year,Description,Cheque Reference No.,Mode,Purpose,Debit,Credit,Balance
0,02-04-2024,4,2024,THE CANDY BAR GURUGRAM,,,,500.00,NaN,
1,02-04-2024,4,2024,PETRO SURCHARGE WAIVER,,,,NaN,20.88,
2,02-04-2024,4,2024,HPCL BANSAL SERVICE ST DELHI,,,,2109.32,NaN,
3,07-04-2024,4,2024,VALUE AND VARIETY NEW DELHI,,,,3475.00,NaN,
4,21-04-2024,4,2024,Netbanking Funds Transfer,,,,NaN,6064.00,
5,21-04-2024,4,2024,BANIYA KI DUKAN PVT LTDNEW DELHI,,,,4403.00,NaN,
6,27-04-2024,4,2024,OM MANNAT DHABA Muzaffarnaga,,,,1046.00,NaN,
7,27-04-2024,4,2024,SATYA FILLING STATION SONEPAT,,,,961.10,NaN,
8,27-04-2024,4,2024,PETRO SURCHARGE WAIVER,,,,NaN,9.52,
9,28-04-2024,4,2024,PETRO SURCHARGE WAIVER,,,,NaN,9.35,



📊 Debit transactions:  64  |  Total: ₹168,519.25
📊 Credit transactions: 25  |  Total: ₹145,971.21
📊 Net spend:           ₹22,548.04
📊 Grand total transactions: 89

📅 Month-wise transaction count:
   2024-04: 13 transactions
   2024-05: 2 transactions
   2024-06: 14 transactions
   2024-07: 8 transactions
   2024-08: 9 transactions
   2024-09: 4 transactions
   2024-10: 7 transactions
   2024-11: 8 transactions
   2024-12: 7 transactions
   2025-01: 6 transactions
   2025-02: 9 transactions
   2025-03: 2 transactions

💾 Saved: 04-2024 to 03 2025 - Deepanshu_Transactions.xlsx

📥 Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Done! Check your downloads.
